In [ ]:
#Import the necessary libraries
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.preprocessing import StandardScaler
from scipy.stats import qmc
import numpy as np
from scipy.optimize import minimize

In [ ]:
X = np.array([
    [0.7281861,  0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.24238435, 0.84409997, 0.5778091,  0.67902128, 0.50195289],
    [0.72952261, 0.7481062,  0.67977464, 0.35655228, 0.67105368],
    [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
    [0.6188123,  0.33180214, 0.18728787, 0.75623847, 0.3288348 ],
    [0.78495809, 0.91068235, 0.7081201,  0.95922543, 0.0049115 ],
    [0.14511079, 0.8966846,  0.89632223, 0.72627154, 0.23627199],
    [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
    [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
    [0.75759436, 0.35583141, 0.0165229,  0.4342072,  0.11243304],
    [0.5367969,  0.30878091, 0.41187929, 0.38822518, 0.5225283 ],
    [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
    [0.6293079,  0.80348368, 0.81140844, 0.04561319, 0.11062446],
    [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
    [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
    [0.25890557, 0.79367771, 0.6421139,  0.19667346, 0.59310318],
    [0.43216593, 0.71561781, 0.3418191,  0.70499988, 0.61496184],
    [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
    [0.9217762,  0.93187122, 0.41487637, 0.59505727, 0.73562569],
    [0.12667892, 0.2914703,  0.06452848, 0.6805146,  0.89281919],
    [0.125720, 0.862724, 0.028544, 0.246605, 0.751206],
    [0.911297, 0.360128, 0.681187, 0.797057, 0.681155],
    [0.7281861,  0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.362369, 0.752276, 0.832593, 0.669698, 0.158606],
    [0.739844, 0.696416, 0.072917, 0.314411, 0.872163],
    [0.885074, 0.601757, 0.716438, 0.837507, 0.396826],
    [0.255160, 0.522740, 0.486545, 0.740267, 0.020766],
    [0.609360, 0.483573, 0.500662, 0.574288, 0.237473],
    [0.393399, 0.196651, 0.599021, 0.857586, 0.016051],
    [0.464984, 0.369388, 0.703268, 0.908164, 0.181040],
    [0.382857, 0.287444, 0.857344, 0.949828, 0.005388],
    [0.425242, 0.406075, 0.602892, 0.950799, 0.394581]

])

y = np.array([
    -0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655, -1.24704893,
    -1.23378638, -1.69434344, -2.57116963, -1.30911635, -1.14478485, -1.91267714,
    -1.62283895, -1.35668211, -2.0184254,  -1.70255784, -1.29424696, -0.93575656,
    -2.15576776, -1.74688209, -2.61310899866408, -1.13914247122897, -0.59258464831999,
    -0.828662161603612, -2.382478286058939, -1.1599256318790203, -0.670451143998971,
    -0.7522686466299074, -0.44168991392979573, -0.2890095251948754, -0.5983826226943812,
    -0.5238375129764865
])

#shape of X & y
print(X.shape)
print(y.shape)

In [ ]:
# Scale the data
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
print(f"\nAfter scaling:")
print(f"X_scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
print(f"y_scaled range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")

# GP Setup
kernel = ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3)) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.1, 100.0)
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=25,
    alpha=0.01,
    normalize_y=False,
    random_state=42
)

# Fit on SCALED data
gpr.fit(X_scaled, y_scaled)

# Check diagnostics
print("\nGP Model Diagnostics:")
# The 'kernel_' attribute is set by the .fit() method after the GPR has been trained.
# If it's missing, it implies the fitting process either failed or was not completed.
if hasattr(gpr, 'kernel_'):
    print(f"  Kernel: {gpr.kernel_}")
    print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
else:
    print("  WARNING: 'kernel_' attribute not found. The Gaussian Process Regressor might not have been fitted successfully or the attribute was not assigned.")
    print(f"  Displaying initial kernel parameters instead:")
    print(f"  Initial Kernel: {gpr.kernel}")
    # Try to access length scales from the initial kernel if it has k2
    if hasattr(gpr.kernel, 'k2') and hasattr(gpr.kernel.k2, 'length_scale'):
        print(f"  Initial Length scales: {gpr.kernel.k2.length_scale}")
print(f"  Training R²: {gpr.score(X_scaled, y_scaled):.3f}")

# Check for overfitting
if gpr.score(X_scaled, y_scaled) > 0.99:
    print("  ⚠️  WARNING: Perfect fit (R²>0.99) suggests overfitting!")
    print("  Consider: more data, higher alpha, or simpler kernel")

# Define bounds for 5D space (use original scale)
bounds = [
    (X[:, 0].min(), X[:, 0].max()),
    (X[:, 1].min(), X[:, 1].max()),
    (X[:, 2].min(), X[:, 2].max()),
    (X[:, 3].min(), X[:, 3].max()),
    (X[:, 4].min(), X[:, 4].max())
]
print(f"\nBounds (original scale): {bounds}")

# Generate 5D candidates (in original scale)
sampler = qmc.LatinHypercube(d=5)
X_candidates = qmc.scale(
    sampler.random(n=15000),  # More candidates for 5D
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
)

# Scale candidates before prediction
X_candidates_scaled = scaler_X.transform(X_candidates)

# Predict on SCALED candidates
y_pred_scaled, y_std_scaled = gpr.predict(X_candidates_scaled, return_std=True)

# Unscale predictions back to original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_std = y_std_scaled * scaler_y.scale_[0]  # Scale uncertainty

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]  # In original scale

print(f"\nNext Point to Sample (original scale):")
print(f"  X = {x_next}")
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}")
    print(f"      pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")

# Sanity checks
print(f"\nSanity Checks:")
print(f"  Predicted y range: [{y_pred.min():.3f}, {y_pred.max():.3f}]")
print(f"  Training y range: [{y.min():.3f}, {y.max():.3f}]")
print(f"  Uncertainty range: [{y_std.min():.3f}, {y_std.max():.3f}]")

if y_std.max() > 10 * np.abs(y.max() - y.min()):
    print("WARNING: Uncertainty is way too high!")
    print("  This suggests the model is very uncertain everywhere")



In [ ]:
#shape of X & y
print(X.shape)
print(y.shape)

Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 2D space
bounds = [
    (X[:, i].min(), X[:, i].max()) for i in range(n_dims)
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )


y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")

Bayessian Optimization

In [ ]:
class BayesianOptimizer:
    def __init__(self, bounds, n_initial=10, kernel=None, alpha=0.01, n_restarts=30):
        """
        Bayesian Optimization for hyperparameter tuning

        Args:
            bounds: List of (min, max) tuples for each dimension
            n_initial: Number of random initial points
            kernel: GP kernel (defaults to Matern)
            alpha: Noise regularization
            n_restarts: GP optimizer restarts
        """
        self.bounds = np.array(bounds)
        self.dim = len(bounds)
        self.n_initial = n_initial
        self.alpha = alpha
        self.n_restarts = n_restarts

        # Initialize scalers
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()

        # Setup kernel
        if kernel is None:
            self.kernel = ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3)) * \
                         Matern(nu=2.5,
                                length_scale=np.ones(self.dim),
                                length_scale_bounds=(0.01, 1000.0))
        else:
            self.kernel = kernel

        # Initialize GP
        self.gpr = GaussianProcessRegressor(
            kernel=self.kernel,
            n_restarts_optimizer=self.n_restarts,
            alpha=self.alpha,
            normalize_y=False,
            random_state=42
        )

        # Storage for observations - Initialize as lists and populate with initial data
        self.X_observed = X.tolist()  # Convert initial numpy array to list
        self.y_observed = y.tolist()  # Convert initial numpy array to list

        # Convert observed lists to numpy arrays for initial scaling and GP fitting
        X_initial_np = np.array(self.X_observed)
        y_initial_np = np.array(self.y_observed).reshape(-1, 1)

        # Fit scalers on the initial data provided
        self.scaler_X.fit(X_initial_np)
        self.scaler_y.fit(y_initial_np)

        # Fit GP on initial scaled data
        X_scaled_initial = self.scaler_X.transform(X_initial_np)
        y_scaled_initial = self.scaler_y.transform(y_initial_np).ravel()
        self.gpr.fit(X_scaled_initial, y_scaled_initial)

        # Print initial diagnostics
        r2 = self.gpr.score(X_scaled_initial, y_scaled_initial)
        print(f"Initial GP Model Fitted with {len(self.y_observed)} observations:")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print("WARNING: Potential overfitting (R²>0.99) on initial data")

    def _get_initial_points(self):
        """Generate initial points using Latin Hypercube Sampling"""
        sampler = qmc.LatinHypercube(d=self.dim, seed=42)
        points = qmc.scale(
            sampler.random(n=self.n_initial),
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )
        return points

    def _acquisition_ucb(self, X, kappa=2.0):
        """Upper Confidence Bound acquisition function"""
        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        y_std = y_std_scaled * self.scaler_y.scale_[0]

        # UCB (negate for minimization)
        return -(y_pred + kappa * y_std)[0]

    def _acquisition_ei(self, X, xi=0.01):
        """Expected Improvement acquisition function"""
        from scipy.stats import norm

        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()[0]
        y_std = y_std_scaled[0] * self.scaler_y.scale_[0]

        # Current best
        y_best = np.max(self.y_observed)

        # Avoid division by zero
        if y_std < 1e-10:
            return 0.0

        # Calculate EI
        z = (y_pred - y_best - xi) / y_std
        ei = (y_pred - y_best - xi) * norm.cdf(z) + y_std * norm.pdf(z)

        return -ei  # Negate for minimization

    def _propose_location(self, acquisition='ucb', kappa=2.0, xi=0.01, n_restarts=25):
        """Propose next sampling point by optimizing acquisition function"""

        # Choose acquisition function
        if acquisition == 'ucb':
            acq_func = lambda x: self._acquisition_ucb(x, kappa=kappa)
        elif acquisition == 'ei':
            acq_func = lambda x: self._acquisition_ei(x, xi=xi)
        else:
            raise ValueError(f"Unknown acquisition function: {acquisition}")

        # Multi-start optimization
        min_val = float('inf')
        min_x = None

        # Generate random starting points
        sampler = qmc.LatinHypercube(d=self.dim, seed=None)
        x0_samples = qmc.scale(
            sampler.random(n=n_restarts), # n_restarts for optimization starts
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )

        for x0 in x0_samples:
            result = minimize(
                acq_func,
                x0=x0,
                bounds=self.bounds,
                method='L-BFGS-B'
            )

            if result.fun < min_val:
                min_val = result.fun
                min_x = result.x

        return min_x

    def update(self, X_new, y_new):
        """Add new observations and refit GP"""
        # Ensure X_new is always treated as a 1D array for single point or a list of 1D arrays for multiple points
        if isinstance(X_new, np.ndarray) and X_new.ndim == 1:
            self.X_observed.append(X_new.tolist()) # Convert to list before appending to self.X_observed list
        elif isinstance(X_new, list) and all(isinstance(x, (list, np.ndarray)) for x in X_new):
            # If X_new is a list of lists/arrays, or a 2D numpy array
            for x_val in X_new:
                self.X_observed.append(x_val.tolist() if isinstance(x_val, np.ndarray) else x_val)
        else:
            # Assuming it's a single point that might not be a numpy array or list itself
            self.X_observed.append(X_new)

        if np.isscalar(y_new):
            self.y_observed.append(y_new)
        elif isinstance(y_new, (list, np.ndarray)):
            self.y_observed.extend(y_new)
        else:
            self.y_observed.append(y_new)

        # Convert observed lists to numpy arrays for scaling and GP fitting
        X_current = np.array(self.X_observed)
        y_current = np.array(self.y_observed).reshape(-1, 1)

        # Scale data - fit_transform on the growing dataset
        X_scaled = self.scaler_X.fit_transform(X_current)
        y_scaled = self.scaler_y.fit_transform(y_current).ravel()

        # Fit GP
        self.gpr.fit(X_scaled, y_scaled)

        # Print diagnostics
        r2 = self.gpr.score(X_scaled, y_scaled)
        print(f"\nGP Model Updated:")
        print(f"  Observations: {len(self.y_observed)}")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print("WARNING: Potential overfitting (R²>0.99)")

    def suggest(self, acquisition='ucb', kappa=2.0, xi=0.01):
        """Suggest next point to evaluate"""
        if len(self.y_observed) < self.n_initial:
            # Use random exploration initially
            # Ensure a distinct initial point is returned each time
            if len(self.X_observed) < self.n_initial:
                new_random_points = self._get_initial_points()
                # Find a point not already in self.X_observed (or similar logic)
                # For simplicity here, just return the next sequential initial point
                return new_random_points[len(self.X_observed)]
            else:
                # If we have enough observed points but not yet hit n_initial for proposal
                # This case is tricky if n_initial is smaller than initial X,y provided
                # Given the fix in __init__ this block will likely not be hit for new random points unless X_observed is explicitly cleared
                return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)
        else:
            # Use acquisition function
            return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)

    def get_best(self):
        """Return best observed point"""
        if len(self.y_observed) == 0:
            return None, None

        best_idx = np.argmax(self.y_observed)
        return np.array(self.X_observed[best_idx]), self.y_observed[best_idx]


In [ ]:
# ==========================
# USAGE EXAMPLE
# ==========================

# Define your 5D bounds
bounds = [
    (0.0, 1.0),   # Parameter 1
    (0.0, 1.0),   # Parameter 2
    (0.0, 1.0),   # Parameter 3
    (0.0, 1.0),   # Parameter 4
    (0.0, 1.0),   # Parameter 5
]

# Initialize optimizer
optimizer = BayesianOptimizer(
    bounds=bounds,
    n_initial=20,      # Initial random samples
    alpha=0.01,        # GP noise
    n_restarts=30      # GP hyperparameter optimization restarts
)

# Define your objective function
def objective_function(params):
    """
    Your black-box function to optimize

    Args:
        params: Array of 5 parameters
    Returns:
        score: Float (higher is better)
    """
    # Example: replace with your actual function
    # e.g., train model with these hyperparameters and return validation score
    score = -np.sum((params - 0.5)**2)  # Dummy function
    return score

# Bayesian Optimization Loop
n_iterations = 50

print("Starting Bayesian Optimization...\n")

for i in range(n_iterations):
    # Get suggestion
    x_next = optimizer.suggest(acquisition='ucb', kappa=2.0)

    # Evaluate objective
    y_next = objective_function(x_next)

    # Update optimizer
    optimizer.update(x_next, y_next)

    # Get current best
    x_best, y_best = optimizer.get_best()

    print(f"Iteration {i+1}/{n_iterations}")
    print(f"  Suggested: {x_next}")
    print(f"  Score: {y_next:.4f}")
    print(f"  Best so far: {y_best:.4f}")
    print(f"  Best params: {x_best}\n")

# Final result
x_best, y_best = optimizer.get_best()
print("\n" + "="*50)
print("OPTIMIZATION COMPLETE")
print("="*50)
print(f"Best score: {y_best:.4f}")
print(f"Best parameters: {x_best}")